<a href="https://colab.research.google.com/github/aszczi/Urban_mobility_in_Cracow/blob/main/Op%C3%B3%C5%BAnienia_Krak%C3%B3w.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analiza Opóźnień Komunikacji Miejskiej w Krakowie (GTFS-RT)

Niniejszy notatnik służy do analizy rzeczywistych opóźnień komunikacji miejskiej w Krakowie. Dane pobierane są w czasie rzeczywistym z usług [GTFS-RT ZTP Kraków](https://gtfs.ztp.krakow.pl/). 

Wykorzystujemy:
- **TripUpdates** (format `.pb` - Protobuf), aby pozyskać estymowane czasy przyjazdów i porównać je do planowanych.
- **Dane statyczne (GTFS)** do podpięcia lokalizacji geo (przystanków) oraz nazw linii.

Notatnik wygeneruje interaktywne mapy ulic ukazujące natężenie opóźnień, a także odpowiednie statystyki i wykresy.

In [ ]:
# Jeśli nie masz ich zainstalowanych, odkomentuj i uruchom poniższą linijkę:
!pip install gtfs-realtime-bindings protobuf requests pandas plotly folium scipy osmnx networkx matplotlib
import requests
from google.transit import gtfs_realtime_pb2
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import TimestampedGeoJson
import osmnx as ox
import networkx as nx
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import datetime
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def fetch_gtfs_rt_delays():
    print("Pobieranie aktualnych danych GTFS-RT (TripUpdates) dla Krakowa...")

    urls = {
        "Tramwaje": "https://gtfs.ztp.krakow.pl/TripUpdates_T.pb",
        "Autobusy": "https://gtfs.ztp.krakow.pl/TripUpdates_A.pb"
    }

    delays_data = []

    for v_type, url in urls.items():
        print(f" Pobieranie danych dla: {v_type}...")
        try:
            feed = gtfs_realtime_pb2.FeedMessage()
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            feed.ParseFromString(response.content)
            
            for entity in feed.entity:
                if entity.HasField('trip_update'):
                    trip_id = entity.trip_update.trip.trip_id
                    route_id = entity.trip_update.trip.route_id
                    
                    for stu in entity.trip_update.stop_time_update:
                        stop_id = stu.stop_id
                        delay = None
                        
                        # Pobieranie opóźnienia z przyjazdu lub odjazdu (preferujemy przyjazd)
                        if stu.HasField('arrival') and stu.arrival.HasField('delay'):
                            delay = stu.arrival.delay
                        elif stu.HasField('departure') and stu.departure.HasField('delay'):
                            delay = stu.departure.delay
                        
                        if delay is not None:
                            arr_time = None
                            if stu.HasField('arrival') and stu.arrival.HasField('time'):
                                arr_time = stu.arrival.time
                            elif stu.HasField('departure') and stu.departure.HasField('time'):
                                arr_time = stu.departure.time
                            
                            delays_data.append({
                                "typ": v_type,
                                "trip_id": trip_id,
                                "line_num": route_id,
                                "stop_sequence": stu.stop_sequence if stu.HasField('stop_sequence') else 0,
                                "stop_id": stop_id,
                                "delay_sec": delay,
                                "delay_min": delay / 60.0,
                                "time": arr_time
                            })
        except Exception as e:
            print(f"  Błąd podczas pobierania {v_type}: {e}")

    df_delays = pd.DataFrame(delays_data)
    
    if not df_delays.empty:
        # Posiadamy delay_sec (opóźnienia dodatnie oznaczają spóźnienie, pomijamy te < 0, bo to znaczy przyspieszenie)
        df_delays = df_delays[df_delays['delay_sec'] > 0]
        
        # Konwersja czasu Uniksowego do datetime i godziny ISO
        if 'time' in df_delays.columns:
            df_delays['datetime'] = pd.to_datetime(df_delays['time'], unit='s')
            df_delays['hour'] = df_delays['datetime'].dt.strftime('%Y-%m-%dT%H:00:00')
            # Jeżeli null (brak time) to przypisujemy bieżącą
            df_delays['hour'] = df_delays['hour'].fillna(datetime.datetime.now().strftime('%Y-%m-%dT%H:00:00'))
        else:
            df_delays['hour'] = datetime.datetime.now().strftime('%Y-%m-%dT%H:00:00')
            
        print(f"\nZakończono. Pobrano {len(df_delays)} rekordów z dodatnimi opóźnieniami.")
    else:
        print("\nNie udało się pobrać żadnych opóźnień lub brak opóźnień w tej chwili.")
        
    return df_delays

df_delays = fetch_gtfs_rt_delays()
df_delays.head()

In [ ]:
# Wczytanie fizycznych lokalizacji przystanków i nazw linii z rozkładów (GTFS Zip)
# Zakładamy, że historyczne (ale w miarę aktualne) pliki przystanków znajdują się lokalnie
static_gtfs_dir = "data/GTFS_ZTP_17.05.26/"
stops_file = os.path.join(static_gtfs_dir, "stops.txt")
routes_file = os.path.join(static_gtfs_dir, "routes.txt")

if os.path.exists(stops_file) and not df_delays.empty:
    df_stops = pd.read_csv(stops_file, dtype=str)
    # Konwersja coords na wartości numeryczne
    df_stops["stop_lat"] = pd.to_numeric(df_stops["stop_lat"], errors="coerce")
    df_stops["stop_lon"] = pd.to_numeric(df_stops["stop_lon"], errors="coerce")
    
    # Łączenie przystanków
    df_merged = df_delays.merge(
        df_stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], 
        on='stop_id', 
        how='inner'
    )
    
    # Przypisywanie nazw linii, jeżeli istnieje routes.txt
    if os.path.exists(routes_file):
        df_routes = pd.read_csv(routes_file, dtype=str)
        df_merged = df_merged.merge(
            df_routes[['route_id', 'route_short_name']], 
            left_on='line_num', 
            right_on='route_id',
            how='left'
        )
        # Zastąpienie wewnętrznego ID linii jej nazwą publiczną np. "152"
        df_merged['linia'] = df_merged['route_short_name'].fillna(df_merged['line_num'])
    else:
        df_merged['linia'] = df_merged['line_num']

    # Obliczanie średniego opóźnienia, maksymalnego, oraz ilości pojazdów per Przystanek
    df_stops_delays = df_merged.groupby(['stop_name', 'stop_lat', 'stop_lon', 'typ'], as_index=False).agg(
        mean_delay_min=('delay_min', 'mean'),
        max_delay_min=('delay_min', 'max'),
        measurements_count=('delay_min', 'count')
    )
    
    # Wyświetlamy tylko te przystanki, przez które opóźnione przejeżdża więcej niż x pojazdów
    df_stops_delays = df_stops_delays[df_stops_delays['measurements_count'] >= 2]
    
    display(df_stops_delays.sort_values(by='mean_delay_min', ascending=False).head())
else:
    print("Brak pliku przystanków lub brak punktów pobranych - upewnij się, ze ścieżka do stops.txt jest poprawna.")

## Połączenie danych GTFS z danymi o ruchu samochodów (Traffic History)
Agregacja obydwu zbiorów w celu wskazania całościowego zatłoczenia w skali całego miasta (niezależnie czy opóźnione jest auto, czy autobus).

In [ ]:
# 1. Wczytujemy historię natężenia drogowego aut
df_cars = pd.read_csv("https://raw.githubusercontent.com/aszczi/Urban_mobility_in_Cracow/refs/heads/main/traffic_history.csv")
df_cars['timestamp'] = pd.to_datetime(df_cars['timestamp'])
df_cars['hour'] = df_cars['timestamp'].dt.strftime('%Y-%m-%dT%H:00:00')
# Samochody mają 'delay_sec', konwertujemy do spójnego 'delay_min' dla autobusów
df_cars['delay_min'] = df_cars['delay_sec'] / 60.0
df_cars['typ'] = 'Samochody'

# Odpowiednie przemianowanie kolumn, aby były zbieżne z df_merged (GTFS)
df_cars_standard = df_cars[['point_name', 'lat', 'lon', 'delay_min', 'hour', 'typ']].copy()

# 2. Przygotowujemy dane GTFS (Komunikacja Miejska) do tego samego formatu
if 'df_merged' in locals() and not df_merged.empty:
    df_gtfs_standard = df_merged[['stop_name', 'stop_lat', 'stop_lon', 'delay_min', 'hour', 'typ']].copy()
    df_gtfs_standard.rename(columns={
        'stop_name': 'point_name',
        'stop_lat': 'lat',
        'stop_lon': 'lon'
    }, inplace=True)
else:
    df_gtfs_standard = pd.DataFrame()

# 3. Złączenie obydwu zestawów w jeden olbrzymi rejestr opóźnień miejskich!
df_all_traffic = pd.concat([df_cars_standard, df_gtfs_standard], ignore_index=True)
df_all_traffic.dropna(subset=['delay_min', 'point_name'], inplace=True)

# Prezentacja wymieszanych danych
df_all_traffic.sample(5)

## Top najgorszych momentów i miejsc (Wykresy globalne)
Rzucamy okiem, o której godzinie miasto osiąga apogeum korków, oraz listujemy czołowe, najwęższe gardła komunikacyjne w systemie (niezależnie czy mówimy o przystanku czy samochodowym skrzyżowaniu).

In [ ]:
# WYKRES 1: Kiedy miasto jest najbardziej zatłoczone (sumarycznie uśrednione w podziale na godziny)
df_hourly = df_all_traffic.groupby('hour', as_index=False).agg(
    avg_delay_min=('delay_min', 'mean'),
    records=('delay_min', 'count')
).sort_values(by='hour')

fig_time = px.line(
    df_hourly, 
    x='hour', 
    y='avg_delay_min', 
    markers=True,
    title="Średnie zatłoczenie miasta Krakowa (opóźnienia w minutach) z podziałem na godziny",
    labels={'hour': 'Godzina', 'avg_delay_min': 'Średnie opóźnienie uczestnika ruchu (min)'}
)
fig_time.show()

# WYKRES 2: Jakie miejsca są najbardziej zatłoczone? - top 20
df_places = df_all_traffic.groupby('point_name', as_index=False).agg(
    avg_delay_min=('delay_min', 'mean'), 
    typ_glowny=('typ',  lambda x: x.mode()[0]) # przeważający typ w danym węźle
)

# Filtrujemy tylko te top 20 najgorszych lokalizacji
top_20_places = df_places.sort_values(by='avg_delay_min', ascending=False).head(20)

fig_places = px.bar(
    top_20_places,
    x='point_name',
    y='avg_delay_min',
    color='typ_glowny',
    title="Top 20 najbardziej opóźnionych / zatłoczonych punktów przestrzeni miejskiej",
    labels={'point_name': 'Nazwa skrzyżowania/przystanku', 'avg_delay_min': 'Średnie opóźnienie trwałego uczestnika (min)'},
    text_auto=':.1f',
    height=600
)
fig_places.update_layout(xaxis_tickangle=-45)
fig_places.show()

## Dynamiczna mapa Całościowego ruchu - Natężenie dróg Kolorem
Teraz rezygnujemy z poszukiwania zaledwie kilku autobusów a nakładamy cały zbiór punktów miejskich. Każdy z rekordów odszukuje optymalną ramkę krawędzi reprezentowanej ulicy, a całość wrzucamy na interaktywną mapę animowaną w czasie za pomocą GeoJSON. Zobaczymy na żywo jak malują się ulice!

In [ ]:
if 'df_all_traffic' in locals() and not df_all_traffic.empty:
    
    # 1. Pobranie grafu drogowego miasta z OSMnx
    lokalizacja = "Kraków, Poland"
    print(f"Pobieranie geometrii dróg dla: {lokalizacja}... ")
    G = ox.graph_from_place(lokalizacja, network_type="drive", simplify=True)
    
    # Do ucinania objętości weźmy uszeregowane "wycinki" dla Foliuma
    # Odchudzamy mapę o zduplikowane nazwy dla celów przypinania krawędzi (nearest_edges)
    unique_all_points = df_all_traffic.drop_duplicates(subset=['point_name', 'lat', 'lon']).copy()
    
    print("Przypinanie zatłoczonych miejsc (GTFS + Auta) do siatki ulic Krakowa (to zajmie kilkanaście sekund)...")
    lats = unique_all_points['lat'].values
    lons = unique_all_points['lon'].values
    
    # Wyszukujemy krawędzie dróg znajdujące się najbliżej punktów natężenia.
    try:
        nearest_edges = ox.nearest_edges(G, X=lons, Y=lats)
    except AttributeError:
        nearest_edges = ox.distance.nearest_edges(G, X=lons, Y=lats)
        
    point_to_edge = dict(zip(unique_all_points['point_name'], nearest_edges))
    
    # Przygotowanie palety kolorów
    features = []
    cmap = plt.get_cmap('RdYlGn_r') 
    norm = mcolors.Normalize(vmin=0, vmax=30) # Skala od 0 do 30 min (żeby odcienie czerwieni nie zakładały apokaliptycznych 3 godzin)
    
    # Przetworzenie całego wolumenu ruchu godzina po godzinie:
    unique_hours = sorted(df_all_traffic['hour'].unique())
    print("Przetwarzanie opóźnień w przedziały godzinowe dla Folium...")
    
    for h in unique_hours:
        df_hour = df_all_traffic[df_all_traffic['hour'] == h]
        
        for idx, row in df_hour.iterrows():
            p_name = row['point_name']
            delay = row['delay_min']
            
            if p_name not in point_to_edge:
                continue
                
            edge = point_to_edge[p_name]
            u, v, key = edge
            
            # Krawędź składa się z dwóch węzłów -> zamieniamy je na LineString
            coords = [[G.nodes[u]['x'], G.nodes[u]['y']], [G.nodes[v]['x'], G.nodes[v]['y']]]
            
            features.append({
                "type": "Feature",
                "geometry": {
                    "type": "LineString",
                    "coordinates": coords
                },
                "properties": {
                    "times": [h] * 2, 
                    "style": {
                        "color": mcolors.to_hex(cmap(norm(delay))),
                        "weight": 8,
                        "opacity": 0.85
                    }
                }
            })
            
    # Generowanie animowanej mapy 
    print("Tworzenie docelowej Interaktywnej Mapy z kolorowaniem ulic z całego miasta...")
    folium_map = folium.Map(location=[50.0614, 19.9383], zoom_start=13, tiles="cartodbdark_matter")
    
    TimestampedGeoJson(
        {"type": "FeatureCollection", "features": features},
        period="PT1H",
        add_last_point=False,
        auto_play=True,
        loop=True,
        max_speed=1,
        loop_button=True,
        time_slider_drag_update=True
    ).add_to(folium_map)
    
    display(folium_map)
else:
    print("Brak scalonej puli danych - wykonaj ramki powyżej.")

## Wykresy (Gdzie są największe opóźnienia)
Przeanalizujmy, które przystanki oraz które linie notują średnio największe opóźnienia w pozyskanej próbce czasowej.

In [ ]:
if 'df_stops_delays' in locals() and not df_stops_delays.empty:
    # Top 15 Przystanków
    top_15_mean = df_stops_delays.sort_values(by='mean_delay_min', ascending=False).head(15)

    fig_bar_stops = px.bar(
        top_15_mean,
        x='stop_name',
        y='mean_delay_min',
        color='typ',
        title="Top 15 przystanków o największym średnim opóźnieniu",
        labels={'stop_name': 'Przystanek', 'mean_delay_min': 'Średnie opóźnienie (min)'},
        text_auto=':.1f',
        height=500
    )
    fig_bar_stops.update_layout(xaxis_tickangle=-45)
    fig_bar_stops.show()
    
if 'df_merged' in locals() and not df_merged.empty:
    # Agregacja po Liniach
    df_route_delays = df_merged.groupby(['linia', 'typ'], as_index=False).agg(
        mean_delay_min=('delay_min', 'mean'),
        measurements_count=('delay_min', 'count')
    )
    
    # Filtrujemy by odrzucić pojedyncze strzały pomiarów dla jakiejś trasy
    df_route_delays = df_route_delays[df_route_delays['measurements_count'] >= 3]
    
    top_15_routes = df_route_delays.sort_values(by='mean_delay_min', ascending=False).head(15)
    
    fig_bar_routes = px.bar(
        top_15_routes,
        x='linia',
        y='mean_delay_min',
        color='typ',
        title="Top 15 linii komunikacyjnych o największym średnim opóźnieniu",
        labels={'linia': 'Numer Linii', 'mean_delay_min': 'Średnie opóźnienie (min)'},
        text_auto=':.1f',
        height=500
    )
    fig_bar_routes.update_layout(xaxis_type='category') # By numery linii zachowywały się jak kategorie
    fig_bar_routes.show()